# Loan Model — Apprentissage complet avec MLflow + predictml-api

Ce notebook couvre l'intégralité du workflow MLOps :

1. Génération et exploration du dataset
2. Construction d'un Pipeline sklearn (ColumnTransformer + GradientBoosting)
3. Tracking complet avec MLflow (metrics, params, tags, artifacts, system metrics)
4. Enregistrement du modèle via l'API predictml
5. Test de prédictions
6. Soumission des résultats observés

**Prérequis** : `docker-compose up -d` doit être lancé.

## 1. Setup & Configuration

In [ ]:
import os
import json
import inspect
import textwrap
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # backend non-interactif pour la génération d'artifacts
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    roc_auc_score, log_loss, confusion_matrix, classification_report,
    RocCurveDisplay
)
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

import mlflow
import mlflow.sklearn
import requests

warnings.filterwarnings('ignore')

print('Imports OK')

In [ ]:
# ── MLflow ──────────────────────────────────────────────────────────────────
MLFLOW_TRACKING_URI = 'http://localhost:5000'
MINIO_ENDPOINT      = 'localhost:9002'
MINIO_ACCESS_KEY    = 'minioadmin'
MINIO_SECRET_KEY    = 'minio_secure_password_123'

os.environ['MLFLOW_S3_ENDPOINT_URL']  = f'http://{MINIO_ENDPOINT}'
os.environ['AWS_ACCESS_KEY_ID']       = MINIO_ACCESS_KEY
os.environ['AWS_SECRET_ACCESS_KEY']   = MINIO_SECRET_KEY

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

# ── API predictml ────────────────────────────────────────────────────────────
API_URL   = 'http://localhost:8000'
API_TOKEN = 'ZC_W_-mcw-01l5W5fN8VFx-h4WornlnxwAtiQutT2BA'
HEADERS   = {'Authorization': f'Bearer {API_TOKEN}'}

# ── Modèle ────────────────────────────────────────────────────────────────────
MODEL_NAME    = 'loan_model'
MODEL_VERSION = '1.0.0'
EXPERIMENT    = MODEL_NAME

print(f'MLflow  : {MLFLOW_TRACKING_URI}')
print(f'API     : {API_URL}')
print(f'Modèle  : {MODEL_NAME} v{MODEL_VERSION}')

## 2. Dataset — Prêt bancaire synthétique

In [ ]:
rng = np.random.default_rng(42)
N   = 500

df = pd.DataFrame({
    'age':          rng.integers(22, 65, N).astype(float),
    'income':       rng.integers(20_000, 120_000, N).astype(float),
    'loan_amount':  rng.integers(5_000, 50_000, N).astype(float),
    'employment':   rng.choice(['employed', 'self_employed', 'unemployed'], N),
    'education':    rng.choice(['high_school', 'bachelor', 'master', 'phd'], N),
    'credit_score': rng.choice(['good', 'fair', 'poor'], N),
})

# Règle de décision déterministe + bruit
score = (
    df['income'] / 120_000 * 0.40
    + np.where(df['credit_score'] == 'good',  0.40,
      np.where(df['credit_score'] == 'fair',  0.20, 0.00))
    + np.where(df['education'].isin(['master', 'phd']), 0.20, 0.10)
    + np.where(df['employment'] == 'employed', 0.10,
      np.where(df['employment'] == 'self_employed', 0.05, 0.00))
    + rng.uniform(-0.08, 0.08, N)
)
df['approved'] = (score > 0.50).astype(int)

print(f'Shape   : {df.shape}')
print(f'Taux approbation : {df["approved"].mean():.1%}')
df.head()

In [ ]:
# EDA rapide
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, col in zip(axes, ['income', 'age', 'loan_amount']):
    for label, grp in df.groupby('approved')[col]:
        ax.hist(grp, alpha=0.6, bins=25, label=f'approved={label}')
    ax.set_title(col)
    ax.legend()

plt.suptitle('Distribution des features numériques par classe', y=1.02)
plt.tight_layout()
plt.show()

print('\nDistribution credit_score × approved :')
print(pd.crosstab(df['credit_score'], df['approved'], normalize='index').round(2))

In [ ]:
# Split train / test
NUMERIC_COLS      = ['age', 'income', 'loan_amount']
CATEGORICAL_COLS  = ['employment', 'education', 'credit_score']
FEATURE_COLS      = NUMERIC_COLS + CATEGORICAL_COLS
TARGET            = 'approved'

X = df[FEATURE_COLS]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f'Train : {X_train.shape}  |  Test : {X_test.shape}')

## 3. Pipeline sklearn

In [ ]:
# Hyperparamètres
GBC_PARAMS = {
    'n_estimators':   150,
    'max_depth':        4,
    'learning_rate':  0.08,
    'subsample':       0.9,
    'min_samples_leaf': 5,
    'random_state':    42,
}

preprocessor = ColumnTransformer([
    ('num', StandardScaler(),                                          NUMERIC_COLS),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), CATEGORICAL_COLS),
])

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier',   GradientBoostingClassifier(**GBC_PARAMS)),
])

pipeline.fit(X_train, y_train)
print('Pipeline entraîné.')

## 4. MLflow — Run complet

In [ ]:
# Calcul des métriques
y_pred      = pipeline.predict(X_test)
y_proba     = pipeline.predict_proba(X_test)[:, 1]

metrics = {
    'accuracy':          float(accuracy_score(y_test, y_pred)),
    'f1_weighted':       float(f1_score(y_test, y_pred, average='weighted')),
    'precision_weighted':float(precision_score(y_test, y_pred, average='weighted')),
    'recall_weighted':   float(recall_score(y_test, y_pred, average='weighted')),
    'roc_auc':           float(roc_auc_score(y_test, y_proba)),
    'log_loss':          float(log_loss(y_test, y_proba)),
}

# Cross-validation accuracy
cv_scores = cross_val_score(pipeline, X, y, cv=5, scoring='accuracy')
metrics['cv_accuracy_mean'] = float(cv_scores.mean())
metrics['cv_accuracy_std']  = float(cv_scores.std())

print('Métriques :')
for k, v in metrics.items():
    print(f'  {k:<28} {v:.4f}')

In [ ]:
# Activer le logging des métriques système (CPU, RAM, etc.)
try:
    mlflow.enable_system_metrics_logging()
    print('System metrics logging activé.')
except Exception as e:
    print(f'System metrics non disponible : {e}')

In [ ]:
mlflow.set_experiment(EXPERIMENT)

with mlflow.start_run(run_name=f'{MODEL_NAME}_v{MODEL_VERSION}') as run:

    # ── Paramètres ────────────────────────────────────────────────────────────
    mlflow.log_params({
        # Hyperparamètres GBC
        **{f'gbc_{k}': v for k, v in GBC_PARAMS.items()},
        # Pipeline
        'preprocessing':        'ColumnTransformer(StandardScaler + OneHotEncoder)',
        'numeric_features':     ', '.join(NUMERIC_COLS),
        'categorical_features': ', '.join(CATEGORICAL_COLS),
        # Dataset
        'dataset':              'synthetic_loan',
        'n_samples':            N,
        'n_features':           len(FEATURE_COLS),
        'test_size':            0.20,
        'stratify':             True,
        'random_state':         42,
    })

    # ── Métriques ─────────────────────────────────────────────────────────────
    mlflow.log_metrics(metrics)

    # ── Tags ──────────────────────────────────────────────────────────────────
    mlflow.set_tags({
        'description':   'Prédiction approbation de prêt bancaire — features mixtes (num + cat)',
        'environment':   'development',
        'author':        'data-scientist',
        'framework':     'scikit-learn',
        'model_type':    'GradientBoostingClassifier',
        'dataset_name':  'synthetic_loan',
        'task':          'binary_classification',
        'target':        'loan_approved',
        'pipeline':      'ColumnTransformer+GBC',
    })

    # ── Evaluation table ──────────────────────────────────────────────────────
    report_dict = classification_report(y_test, y_pred, output_dict=True)
    report_df   = pd.DataFrame(report_dict).T.reset_index().rename(columns={'index': 'class'})
    mlflow.log_table(data=report_df, artifact_file='evaluation/classification_report.json')

    # Table des prédictions sur le test set
    eval_table = X_test.copy()
    eval_table['y_true']  = y_test.values
    eval_table['y_pred']  = y_pred
    eval_table['y_proba'] = y_proba.round(4)
    mlflow.log_table(data=eval_table.reset_index(drop=True), artifact_file='evaluation/predictions_test.json')

    # ── Artifact : rapport HTML ────────────────────────────────────────────────
    fig_html, axes_html = plt.subplots(1, 3, figsize=(18, 5))

    # Courbe AUC-ROC
    RocCurveDisplay.from_predictions(y_test, y_proba, ax=axes_html[0], name=MODEL_NAME)
    axes_html[0].set_title('Courbe AUC-ROC')
    axes_html[0].plot([0, 1], [0, 1], 'k--', alpha=0.5)

    # Matrice de confusion
    cm = confusion_matrix(y_test, y_pred)
    im = axes_html[1].imshow(cm, interpolation='nearest', cmap='Blues')
    axes_html[1].set_title('Matrice de confusion')
    axes_html[1].set_xlabel('Prédit')
    axes_html[1].set_ylabel('Réel')
    for i in range(2):
        for j in range(2):
            axes_html[1].text(j, i, cm[i, j], ha='center', va='center',
                              color='white' if cm[i, j] > cm.max() / 2 else 'black', fontsize=14)
    plt.colorbar(im, ax=axes_html[1])

    # Feature importance
    gbc         = pipeline.named_steps['classifier']
    ohe_cats    = pipeline.named_steps['preprocessor'].named_transformers_['cat'] \
                      .get_feature_names_out(CATEGORICAL_COLS).tolist()
    feat_names  = NUMERIC_COLS + ohe_cats
    importances = gbc.feature_importances_
    top_idx     = np.argsort(importances)[-12:]
    axes_html[2].barh([feat_names[i] for i in top_idx], importances[top_idx])
    axes_html[2].set_title('Feature Importance (top 12)')
    axes_html[2].set_xlabel('Importance')

    plt.tight_layout()

    report_path = '/tmp/loan_model_report.png'
    fig_html.savefig(report_path, dpi=120, bbox_inches='tight')
    plt.close(fig_html)
    mlflow.log_artifact(report_path, artifact_path='evaluation')

    # ── Artifact : rapport HTML complet ───────────────────────────────────────
    html_content = f"""<!DOCTYPE html>
<html lang='fr'>
<head><meta charset='utf-8'><title>Loan Model Report</title>
<style>
  body {{ font-family: Arial, sans-serif; margin: 40px; background: #f9f9f9; }}
  h1 {{ color: #2c3e50; }} h2 {{ color: #34495e; }}
  table {{ border-collapse: collapse; width: 100%; margin: 10px 0; }}
  th, td {{ border: 1px solid #ddd; padding: 8px; text-align: left; }}
  th {{ background: #3498db; color: white; }}
  tr:nth-child(even) {{ background: #f2f2f2; }}
  .metric {{ display: inline-block; background: #3498db; color: white;
             border-radius: 6px; padding: 8px 16px; margin: 4px; font-size: 14px; }}
  .metric span {{ display: block; font-size: 22px; font-weight: bold; }}
</style></head>
<body>
<h1>Rapport — {MODEL_NAME} v{MODEL_VERSION}</h1>
<p><b>Description :</b> Prédiction d'approbation de prêt bancaire — features mixtes (num + cat)</p>
<p><b>Dataset :</b> synthetic_loan — {N} observations, {len(FEATURE_COLS)} features</p>
<p><b>Run MLflow :</b> {run.info.run_id}</p>

<h2>Métriques</h2>
<div>
  {''.join(f'<div class="metric">{k.replace("_", " ")}<span>{v:.4f}</span></div>' for k, v in metrics.items())}
</div>

<h2>Rapport de classification</h2>
{report_df.to_html(index=False, float_format=lambda x: f'{x:.3f}')}

<h2>Paramètres du modèle</h2>
<table>
  <tr><th>Paramètre</th><th>Valeur</th></tr>
  {''.join(f'<tr><td>{k}</td><td>{v}</td></tr>' for k, v in GBC_PARAMS.items())}
</table>

<h2>Features</h2>
<p><b>Numériques :</b> {', '.join(NUMERIC_COLS)}</p>
<p><b>Catégorielles :</b> {', '.join(CATEGORICAL_COLS)}</p>

<h2>Top 5 features les plus importantes</h2>
<table>
  <tr><th>Feature</th><th>Importance</th></tr>
  {''.join(f'<tr><td>{feat_names[i]}</td><td>{importances[i]:.4f}</td></tr>' for i in np.argsort(importances)[-5:][::-1])}
</table>
</body></html>"""

    html_path = '/tmp/loan_model_report.html'
    with open(html_path, 'w', encoding='utf-8') as f:
        f.write(html_content)
    mlflow.log_artifact(html_path, artifact_path='evaluation')

    # ── Artifact : code source ─────────────────────────────────────────────────
    source_code = textwrap.dedent(f"""
        # Code source — loan_model v{MODEL_VERSION}
        # Généré automatiquement depuis le notebook loan_model_training.ipynb

        import pandas as pd
        import numpy as np
        from sklearn.compose import ColumnTransformer
        from sklearn.ensemble import GradientBoostingClassifier
        from sklearn.pipeline import Pipeline
        from sklearn.preprocessing import OneHotEncoder, StandardScaler

        NUMERIC_COLS     = {NUMERIC_COLS}
        CATEGORICAL_COLS = {CATEGORICAL_COLS}

        preprocessor = ColumnTransformer([
            ('num', StandardScaler(),                                              NUMERIC_COLS),
            ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False),   CATEGORICAL_COLS),
        ])

        pipeline = Pipeline([
            ('preprocessor', preprocessor),
            ('classifier',   GradientBoostingClassifier(
                n_estimators={GBC_PARAMS['n_estimators']},
                max_depth={GBC_PARAMS['max_depth']},
                learning_rate={GBC_PARAMS['learning_rate']},
                subsample={GBC_PARAMS['subsample']},
                min_samples_leaf={GBC_PARAMS['min_samples_leaf']},
                random_state={GBC_PARAMS['random_state']},
            )),
        ])

        pipeline.fit(X_train, y_train)
    """)

    code_path = '/tmp/loan_model_source.py'
    with open(code_path, 'w') as f:
        f.write(source_code)
    mlflow.log_artifact(code_path, artifact_path='source')

    # ── Enregistrement du modèle dans MLflow ──────────────────────────────────
    mlflow.sklearn.log_model(
        pipeline,
        artifact_path='model',
        registered_model_name=MODEL_NAME,
    )

    RUN_ID = run.info.run_id
    print(f'Run ID  : {RUN_ID}')
    print(f'MLflow  : {MLFLOW_TRACKING_URI}/#/experiments/{run.info.experiment_id}/runs/{RUN_ID}')
    print(f'Accuracy: {metrics["accuracy"]:.4f}  |  AUC: {metrics["roc_auc"]:.4f}')

## 5. Enregistrement via l'API predictml

In [ ]:
payload = {
    'name':             MODEL_NAME,
    'version':          MODEL_VERSION,
    'description':      'Prédiction approbation de prêt bancaire — features mixtes (num + cat)',
    'algorithm':        'GradientBoostingClassifier',
    'mlflow_run_id':    RUN_ID,
    'accuracy':         metrics['accuracy'],
    'f1_score':         metrics['f1_weighted'],
    'features_count':   len(FEATURE_COLS),
    'classes':          json.dumps([0, 1]),
    'training_params':  json.dumps({**GBC_PARAMS,
                                    'preprocessing': 'ColumnTransformer',
                                    'numeric_features': NUMERIC_COLS,
                                    'categorical_features': CATEGORICAL_COLS}),
    'training_dataset': 'synthetic_loan',
}

resp = requests.post(f'{API_URL}/models', headers=HEADERS, data=payload)

if resp.status_code == 201:
    model_data = resp.json()
    print(f'Modèle créé — id={model_data["id"]}')
elif resp.status_code == 409:
    print('Modèle existe déjà (409) — continuons.')
else:
    print(f'Erreur {resp.status_code}: {resp.text}')
    raise RuntimeError('Échec création modèle')

In [ ]:
# Vérifier les métadonnées du modèle
resp = requests.get(f'{API_URL}/models/{MODEL_NAME}/{MODEL_VERSION}', headers=HEADERS)
model_meta = resp.json()

print(f'Nom         : {model_meta["name"]} v{model_meta["version"]}')
print(f'Description : {model_meta["description"]}')
print(f'Accuracy    : {model_meta["accuracy"]}')
print(f'run_id      : {model_meta["mlflow_run_id"]}')
print(f'Features    : {model_meta["feature_names"]}')

In [ ]:
# Passer la version en production
resp = requests.patch(
    f'{API_URL}/models/{MODEL_NAME}/{MODEL_VERSION}',
    headers=HEADERS,
    json={'is_production': True},
)
print(f'is_production : {resp.json()["is_production"]}')

## 6. Test de prédictions

In [ ]:
# Profil 1 — bon candidat
pred1 = requests.post(f'{API_URL}/predict', headers=HEADERS, json={
    'model_name': MODEL_NAME,
    'id_obs':     'client-001',
    'features': {
        'age':          35.0,
        'income':       85_000.0,
        'loan_amount':  15_000.0,
        'employment':   'employed',
        'education':    'master',
        'credit_score': 'good',
    }
}).json()

# Profil 2 — candidat risqué
pred2 = requests.post(f'{API_URL}/predict', headers=HEADERS, json={
    'model_name': MODEL_NAME,
    'id_obs':     'client-002',
    'features': {
        'age':          28.0,
        'income':       22_000.0,
        'loan_amount':  45_000.0,
        'employment':   'unemployed',
        'education':    'high_school',
        'credit_score': 'poor',
    }
}).json()

# Profil 3 — version explicite
pred3 = requests.post(f'{API_URL}/predict', headers=HEADERS, json={
    'model_name':    MODEL_NAME,
    'model_version': MODEL_VERSION,
    'id_obs':        'client-003',
    'features': {
        'age':          45.0,
        'income':       60_000.0,
        'loan_amount':  25_000.0,
        'employment':   'self_employed',
        'education':    'bachelor',
        'credit_score': 'fair',
    }
}).json()

for label, pred in [('Bon candidat (client-001)', pred1),
                     ('Risqué (client-002)',       pred2),
                     ('Intermédiaire (client-003)', pred3)]:
    decision = 'APPROUVÉ' if pred.get('prediction') == 1 else 'REFUSÉ'
    probas = pred.get('probability', [])
    prob_str = f" | P(refus)={probas[0]:.2f}, P(approbation)={probas[1]:.2f}" if probas else ''
    print(f'{label} → {decision}{prob_str}')

In [ ]:
# Vérifier les prédictions stockées en base
from datetime import date
today = date.today().isoformat()

resp = requests.get(
    f'{API_URL}/predictions',
    headers=HEADERS,
    params={'model_name': MODEL_NAME, 'start': f'{today}T00:00:00', 'limit': 10}
)
data = resp.json()
print(f"Total prédictions stockées aujourd'hui : {data['total']}")
for p in data['predictions']:
    print(f"  id_obs={p['id_obs']}  →  {p['prediction_result']}  ({p['response_time_ms']:.1f}ms)")

## 7. Observed Results — Résultats réels

In [ ]:
# Simulation : quelques semaines plus tard, on connaît les vrais résultats
observed = [
    {'id_obs': 'client-001', 'model_name': MODEL_NAME, 'observed_result': 1},  # bien approuvé, a remboursé
    {'id_obs': 'client-002', 'model_name': MODEL_NAME, 'observed_result': 0},  # bien refusé
    {'id_obs': 'client-003', 'model_name': MODEL_NAME, 'observed_result': 1},  # approuvé mais a fait défaut
]

resp = requests.post(
    f'{API_URL}/observed_results',
    headers=HEADERS,
    json=observed
)
print(f'Status : {resp.status_code}')
print(json.dumps(resp.json(), indent=2))

In [ ]:
# Récupérer les observed results pour ce modèle
resp = requests.get(
    f'{API_URL}/observed_results',
    headers=HEADERS,
    params={'model_name': MODEL_NAME}
)
data = resp.json()
print(f'Total : {data["total"]}')
for r in data['results']:
    print(f"  id_obs={r['id_obs']}  →  observed={r['observed_result']}")

## Récapitulatif

| Étape | Status |
|-------|--------|
| Dataset généré (500 obs, 6 features) | ✅ |
| Pipeline ColumnTransformer + GBC entraîné | ✅ |
| Run MLflow — métriques, params, tags | ✅ |
| Evaluation tables loggées | ✅ |
| Artifacts (PNG + HTML rapport + code source) | ✅ |
| System metrics | ✅ |
| Modèle enregistré via POST /models | ✅ |
| Version passée en production | ✅ |
| Prédictions testées avec id_obs | ✅ |
| Observed results soumis | ✅ |